# Attempting a more efficient approach to the shortest paths problem.

The naive solution in `main.ipynb` calls `nx.algorithms.all_shortest_paths()` for each source-target pair - a total of `len(source) * len(target)` searches. This does not scale well, so to make this faster we want to limit the number of BFS searches. We also need to collect the interneurons, so our pipeline cannot give us path lengths alone.

There are two competing methods I want to test (with the algorithm outline worked out with the help of ChatGPT). I will outline this using ORN -> OviDN as an example:

1. Single-source BFS

- Run a BFS for each source neuron (e.g. 1 BFS for each yeast ORN = ~560 BFS).
- Reverse the edges in the connectome, then run a BFS on that for each target neuron (e.g. 16 BFS for 16 OviDNs)
- Keep every edge `u -> v` that satisfies the following condition:
  - `dist(source -> u) + 1 + dist(v -> target) = D`, where `D = dist(source, target)`
- Compose the edge list, then aggregate to group level like normal

1. Multi-source BFS

- Run a multi-source BFS of all source neurons, which will return the distance from each node in the connectome to its closest source
- Run a multi-source BFS of all target neurons using the reversed graph, which will return the distance from each node in the connectome to its nearest target
- Somehow reconstruct "edge list" from this, with the caveat that you lose some information about which source/target a given node is closest to.
  - This method needs some more refinement.

In [2]:
import pandas as pd
import numpy as np
from scipy import sparse

edge_list = ( # edges are separated by neuropil, but we only care about total synapse count. group by edges and sum synapse counts
    pd.read_csv('connections_princeton.csv')
    .drop(columns=['nt_type']) # all empty
    .groupby(['pre_root_id', 'post_root_id'], as_index=False)['syn_count']
    .sum()
)

all_neurons = pd.concat([edge_list['pre_root_id'], edge_list['post_root_id']]).unique()
id_to_idx = {root_id: i for i, root_id in enumerate(all_neurons)}
# idx_to_id = {i: root_id for i, root_id in enumerate(id_to_idx)}
N = len(all_neurons)

pre_root_ids = edge_list['pre_root_id'].map(id_to_idx).values.astype(np.int32)
post_root_ids = edge_list['post_root_id'].map(id_to_idx).values.astype(np.int32)

G = sparse.csr_matrix(((np.ones(shape=len(pre_root_ids), dtype=bool)), (pre_root_ids, post_root_ids)), shape=(N, N))
G_rev = G.transpose().tocsr()


In [3]:
def bfs(graph, source):
    dist_arr = np.full(graph.shape[0], -1, np.int32)
    current = np.zeros(graph.shape[0], bool)

    dist_arr[source] = 0
    current[source] = True
    depth = 0

    while current.any():
        depth += 1
        next = np.asarray(current @ graph).ravel() > 0 
        next = next & (dist_arr == -1)

        if not next.any():
            break

        dist_arr[next] = depth
        current = next

    return dist_arr

In [6]:
def shortest_paths(source, target):
    source_indices = np.array([id_to_idx[s] for s in source if s in id_to_idx])
    target_indices = np.array([id_to_idx[t] for t in target if t in id_to_idx])

    is_looping_source = len(source_indices) <= len(target_indices) # find the smaller neuron population to reduce number of BFS calls

    small_indices = source_indices if is_looping_source else target_indices
    large_indices = target_indices if is_looping_source else source_indices
    union_mask = np.zeros(len(pre_root_ids), bool)

    for s in small_indices:
        if is_looping_source:
            distance = bfs(G, np.array([[s]]))
            valid = (distance[pre_root_ids] >= 0) & (distance[post_root_ids] >= 0) & (distance[post_root_ids] == distance[pre_root_ids] + 1)
        else:
            distance = bfs(G_rev, np.array([[s]]))
            valid = (distance[pre_root_ids] >= 0) & (distance[post_root_ids] >= 0) & (distance[pre_root_ids] == distance[post_root_ids] + 1)

        dag_pre_ids = pre_root_ids[valid]
        dag_post_ids = post_root_ids[valid]

        G_dag = sparse.csr_matrix((np.ones(len(dag_pre_ids), bool), (dag_pre_ids, dag_post_ids)), shape=(N, N))

        if is_looping_source:
            reach = bfs(G_dag.transpose().tocsr(), large_indices) >= 0
            keep = reach[dag_post_ids]
        else:
            reach = bfs(G_dag, large_indices) >= 0
            keep = reach[dag_pre_ids]

        edge_ok = valid.copy()
        edge_ok[valid] = keep

        union_mask |= edge_ok

    return edge_list[union_mask].copy()

In [7]:
import pandas as pd

# Define the neuron populations
given_neurons = pd.read_csv('given_neurons.csv')
oviDNs = given_neurons[given_neurons['group'] == 'OviDN']['root_id'].tolist()
CAs = given_neurons[given_neurons['group'] == 'CA']['root_id'].tolist()
ORNs = given_neurons[given_neurons['group'] == 'ORN']['root_id'].tolist()

# Define the combinations of source and target groups
combinations = [
    (ORNs, oviDNs, 'ORN_to_OviDN'),
    (oviDNs, ORNs, 'OviDN_to_ORN'),
    (oviDNs, CAs, 'OviDN_to_CA'),
    (CAs, oviDNs, 'CA_to_OviDN'),
    (ORNs, CAs, 'ORN_to_CA'),
    (CAs, ORNs, 'CA_to_ORN')
]

# Initialize an empty list to store all shortest path edges
all_shortest_path_edges = []

# Iterate over each combination and get the shortest path edges
for source, target, label in combinations:
    edges = shortest_paths(source, target)
    edges['direction'] = label  # Add a column indicating the direction
    all_shortest_path_edges.append(edges)

# Concatenate all shortest path edges into a single DataFrame
master_edge_list = pd.concat(all_shortest_path_edges, ignore_index=True)

# Save the master edge list to a CSV file (optional)
master_edge_list.to_csv('master_edge_list_with_direction.csv', index=False)

# Display the master edge list
print(master_edge_list)

               pre_root_id        post_root_id  syn_count     direction
0       720575941343170595  720575941439824402          3  ORN_to_OviDN
1       720575941343170595  720575941441694555          5  ORN_to_OviDN
2       720575941343170595  720575941484789178          3  ORN_to_OviDN
3       720575941343170595  720575941492598476          5  ORN_to_OviDN
4       720575941343170595  720575941504989015          3  ORN_to_OviDN
...                    ...                 ...        ...           ...
163181  720575941732487211  720575941551860735          4     CA_to_ORN
163182  720575941733663275  720575941556925779          4     CA_to_ORN
163183  720575941733663275  720575941643582821          4     CA_to_ORN
163184  720575941734311723  720575941466904342          4     CA_to_ORN
163185  720575941734311723  720575941560552563          3     CA_to_ORN

[163186 rows x 4 columns]


## Attempt at enrichment analysis

In [12]:
# Load neuron attributes
neurons = pd.read_csv('neurons.csv')
serotonergic_neurons = neurons[neurons['Predicted NT type'] == 'SER']

# Total number of serotonergic neurons in the graph
total_serotonergic = len(serotonergic_neurons)

# Load the master edge list with direction information
master_edge_list = pd.read_csv('master_edge_list_with_direction.csv')

# Define combinations of source and target groups
combinations = [
    (ORNs, oviDNs, 'ORN_to_OviDN'),
    (oviDNs, ORNs, 'OviDN_to_ORN'),
    (oviDNs, CAs, 'OviDN_to_CA'),
    (CAs, oviDNs, 'CA_to_OviDN'),
    (ORNs, CAs, 'ORN_to_CA'),
    (CAs, ORNs, 'CA_to_ORN')
]

# Initialize a dictionary to store results
results = {}

# Iterate over each combination and calculate enrichment
for source, target, label in combinations:
    edges = master_edge_list[master_edge_list['direction'] == label]
    
    # Find serotonergic neurons on the path
    serotonergic_on_path = edges.merge(serotonergic_neurons, left_on='pre_root_id', right_on='Root ID', how='inner')
    serotonergic_on_path_count = len(serotonergic_on_path)
    
    # Calculate enrichment for the entire path
    fraction_on_path = serotonergic_on_path_count / len(edges)
    fraction_in_graph = total_serotonergic / len(all_neurons)
    enrichment_overall = fraction_on_path / fraction_in_graph
    
    # Store results
    results[label] = {
        'fraction_on_path': fraction_on_path,
        'fraction_in_graph': fraction_in_graph,
        'enrichment_overall': enrichment_overall
    }

for r in results.keys():
    print(f"{r}: {results[r]['enrichment_overall']}")

ORN_to_OviDN: 0.31488873173147763
OviDN_to_ORN: 0.9520254514349956
OviDN_to_CA: 2.612902503632535
CA_to_OviDN: 29.830497297664774
ORN_to_CA: 0.405620702389091
CA_to_ORN: 2.414014619271129


## Comparison of output

In [13]:
import pandas as pd

# Load your master edge list with direction information
your_edge_list = pd.read_csv('master_edge_list_with_direction.csv')
your_relevant_edges = your_edge_list[['direction', 'pre_root_id', 'post_root_id']]

# Load your colleague's edge list
colleague_edge_list = pd.read_csv('/Users/ayushshrivastava/Downloads/ORN_CA_oviDN_COMPLETE_package/MASTER_pathway_edges_v2.csv')
colleague_relevant_edges = colleague_edge_list[['direction', 'pre_root_id', 'post_root_id']]

# Convert the relevant edges to a set of tuples for easy comparison
your_unique_edges = set(your_relevant_edges.apply(tuple, axis=1))
colleague_unique_edges = set(colleague_relevant_edges.apply(tuple, axis=1))

# Find common and unique edges
common_edges = your_unique_edges & colleague_unique_edges
your_only_edges = your_unique_edges - colleague_unique_edges
colleague_only_edges = colleague_unique_edges - your_unique_edges

# Display the results
print("Number of common edges:", len(common_edges))
print("Number of edges in your edge list but not in colleague's edge list:", len(your_only_edges))
print("Number of edges in colleague's edge list but not in your edge list:", len(colleague_only_edges))

# Optionally, save the unique edges to CSV files
your_only_edges_df = pd.DataFrame(list(your_only_edges), columns=['direction', 'pre_root_id', 'post_root_id'])
colleague_only_edges_df = pd.DataFrame(list(colleague_only_edges), columns=['direction', 'pre_root_id', 'post_root_id'])

your_only_edges_df.to_csv('unique_to_your_edge_list.csv', index=False)
colleague_only_edges_df.to_csv('unique_to_colleague_edge_list.csv', index=False)

print("Unique edges in your edge list saved to 'unique_to_your_edge_list.csv'")
print("Unique edges in colleague's edge list saved to 'unique_to_colleague_edge_list.csv'")

Number of common edges: 55122
Number of edges in your edge list but not in colleague's edge list: 108064
Number of edges in colleague's edge list but not in your edge list: 420285
Unique edges in your edge list saved to 'unique_to_your_edge_list.csv'
Unique edges in colleague's edge list saved to 'unique_to_colleague_edge_list.csv'
